# Limpeza do CSV do Museu FACOM (Tainacan) — versão para o VS Code

Este notebook pega o CSV exportado do Tainacan e gera **3 tabelas limpas** na pasta `data/` do projeto:

| Arquivo | O que tem |
|---|---|
| `objects.csv` | um registro por objeto |
| `organisations.csv` | um registro por fabricante (coluna *Autoria*) |
| `object_maker.csv` | liga cada objeto ao seu fabricante |

Essas são as tabelas que o `scripts/facom_loader.py` lê.

**Antes de começar (uma vez só)**

1. No VS Code, instale as extensões **Python** e **Jupyter**.
2. No terminal, dentro da pasta do projeto: `.venv/Scripts/python -m pip install pandas ipykernel`
3. No canto superior direito do notebook, clique em **Selecionar Kernel** e escolha o Python da pasta `.venv`.

**Como usar:** coloque o CSV exportado em `bruto/` e rode as células em ordem (`Shift+Enter`), ou use **Executar Tudo**.

## 1. Escolher o arquivo
Ao rodar a célula abaixo, abre uma janela do Windows para você **escolher o CSV** (como o botão "Escolher arquivos" do Colab). A janela pode abrir atrás do VS Code: se nada aparecer, olhe a barra de tarefas.
O arquivo escolhido é copiado para `bruto/`. Se você fechar a janela sem escolher, o notebook usa o CSV mais recente de `bruto/`.

In [ ]:
import re
import shutil
from pathlib import Path

import pandas as pd

# A raiz do projeto é a pasta que contém "bruto/" (sobe a partir de onde o notebook está).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bruto").is_dir())
BRUTO = RAIZ / "bruto"

def escolher_csv():
    """Abre a janela de escolha de arquivo. Devolve o caminho, ou None se cancelar."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        janela = tk.Tk()
        janela.withdraw()
        janela.attributes("-topmost", True)   # traz a janela para frente
        escolhido = filedialog.askopenfilename(
            title="Escolha o CSV exportado do Tainacan",
            initialdir=str(BRUTO),
            filetypes=[("CSV", "*.csv"), ("Todos os arquivos", "*.*")],
        )
        janela.destroy()
        return Path(escolhido) if escolhido else None
    except Exception as erro:
        print("Não consegui abrir a janela de escolha:", erro)
        return None

caminho_csv = escolher_csv()

if caminho_csv is None:
    candidatos = sorted(BRUTO.glob("*.csv"), key=lambda p: p.stat().st_mtime)
    if not candidatos:
        raise SystemExit("Nenhum arquivo escolhido e nenhum .csv em bruto/. Rode a célula de novo.")
    caminho_csv = candidatos[-1]
    print("Nenhum arquivo escolhido; usando o mais recente de bruto/.")
elif caminho_csv.parent.resolve() != BRUTO.resolve():
    destino = BRUTO / caminho_csv.name
    shutil.copy2(caminho_csv, destino)
    print("Cópia salva em", destino)
    caminho_csv = destino

print("Projeto:", RAIZ)
print("Arquivo usado:", caminho_csv.name)

## 2. Ler e olhar o arquivo
O `pandas` é a biblioteca que trabalha com tabelas. `df` (de *DataFrame*) é a tabela na memória.

In [ ]:
df = pd.read_csv(caminho_csv)
print("Linhas e colunas:", df.shape)

# Os nomes das colunas vêm do Tainacan com sufixos tipo "Título|core_title|display_yes".
# Ficamos só com a parte antes do primeiro "|".
df.columns = [c.split("|")[0] for c in df.columns]
print(df.columns.tolist())

## 3. Funções de limpeza
Neste CSV, "sem informação" aparece de várias formas: `s/r` (sem registro), `-`, `s` com um caractere invisível de retorno de linha, ou célula vazia. As funções abaixo transformam tudo isso em **vazio de verdade** e arrumam espaços e pontos finais soltos (`Brasil.` e `Brasil` passam a ser iguais).

In [ ]:
VAZIOS = {"", "-", "s", "s/r", "s\\r", "nan"}

def limpar_texto(valor):
    """Tira \\r e espaços; troca 'sem informação' por vazio."""
    if pd.isna(valor):
        return None
    texto = str(valor).replace("\r", "").strip()
    return None if texto.lower() in VAZIOS else texto

def limpar_curto(valor):
    """Para campos curtos (autoria, origem...): também tira o ponto final."""
    texto = limpar_texto(valor)
    return texto.rstrip(". ").strip() if texto else None

def slug(texto):
    """'Hewlett-Packard (HP)' -> 'hewlett-packard-hp' (usado nos identificadores)."""
    return re.sub(r"[^a-z0-9]+", "-", texto.lower()).strip("-")

## 4. Montar a tabela de objetos

In [ ]:
objetos = pd.DataFrame({
    "id": "obj-" + df["special_item_id"].astype(str),
    "label": df["Título"].map(limpar_texto),
    "description": df["Descrição"].map(limpar_texto),
    "inventory_code": df["Código do inventário"].map(limpar_curto),
    "category": df["Categoria do acervo"].map(limpar_curto),
    "term": df["Termo"].map(limpar_curto),
    "date_attributed": df["Data Atribuída"].map(limpar_curto),
    "origin": df["Origem"].map(limpar_curto),
    "acquisition_mode": df["Modo de aquisição"].map(limpar_curto),
    "maker": df["Autoria"].map(limpar_curto),
    "url": df["public_url"],
})
objetos.head()

## 5. Correções manuais (você decide)
A limpeza automática não sabe que `Dell lnc` é um erro de digitação. Veja os valores distintos e ajuste os dicionários abaixo se quiser.

In [ ]:
print("Fabricantes (Autoria):")
for nome in sorted(objetos["maker"].dropna().unique()):
    print("  -", nome)
print()
print("Categorias:")
for nome in sorted(objetos["category"].dropna().unique()):
    print("  -", nome)

In [ ]:
# Formato: "texto como está" : "texto corrigido"
CORRIGIR_FABRICANTE = {
    "Dell lnc": "Dell Inc",          # "lnc" (L minúsculo) é erro de digitação de "Inc"
    # Atenção: nomes com "/" ou "Fabricado para" (ex.: "IBM / Lenovo") NÃO foram
    # juntados com outro fabricante de propósito. Confira o objeto no acervo antes de decidir.
}

CORRIGIR_CATEGORIA = {
    "Máquinas de Escrever": "Máquina de Escrever",   # plural x singular
}

objetos["maker"] = objetos["maker"].replace(CORRIGIR_FABRICANTE)
objetos["category"] = objetos["category"].replace(CORRIGIR_CATEGORIA)

## 6. Organizações e tabela de ligação
Cada fabricante distinto vira **um** registro de organização, e uma tabela separada diz qual objeto foi feito por qual organização. É o mesmo desenho do SMG descrito no artigo (tabelas de objetos, organizações e uma tabela de junção).

In [ ]:
nomes = sorted(objetos["maker"].dropna().unique())
organizacoes = pd.DataFrame({
    "id": ["org-" + slug(n) for n in nomes],
    "label": nomes,
})

mapa = dict(zip(organizacoes["label"], organizacoes["id"]))
juncao = objetos.dropna(subset=["maker"])[["id", "maker"]].copy()
juncao["org_id"] = juncao["maker"].map(mapa)
juncao = juncao[["id", "org_id"]].rename(columns={"id": "object_id"})

print(len(objetos), "objetos |", len(organizacoes), "organizações |", len(juncao), "relações")
organizacoes

## 7. Salvar em `data/`
Grava os 3 CSVs na pasta `data/`. Se já existirem tabelas lá, uma cópia delas vai para `data/anterior/` antes de serem substituídas.

In [ ]:
SAIDA = RAIZ / "data"
SAIDA.mkdir(exist_ok=True)
ARQUIVOS = ["objects.csv", "organisations.csv", "object_maker.csv"]

if any((SAIDA / a).exists() for a in ARQUIVOS):
    ANTERIOR = SAIDA / "anterior"
    ANTERIOR.mkdir(exist_ok=True)
    for a in ARQUIVOS:
        if (SAIDA / a).exists():
            shutil.copy2(SAIDA / a, ANTERIOR / a)
    print("Cópia das tabelas anteriores em", ANTERIOR)

objetos.drop(columns=["maker"]).to_csv(SAIDA / "objects.csv", index=False)
organizacoes.to_csv(SAIDA / "organisations.csv", index=False)
juncao.to_csv(SAIDA / "object_maker.csv", index=False)

print("Tabelas salvas em", SAIDA)
print()
print("Próximos passos:")
print("  1. Se há fabricantes novos, adicione-os ao data/sameas.csv (o buscar_candidatos.py")
print("     não altera um sameas.csv que já existe) e preencha o wikidata_id.")
print("  2. Rode o pipeline na ordem: facom_loader.py -> buscar_propriedades.py ->")
print("     es_para_fuseki.py -> grafo_html.py")